# W10 tutorial — the Validation File

**ACFI20009 Financial Data — 26/27**  
Dr Zeming Li, University of Bristol

**This week you build the Validation File:** the document you would attach to a model-risk review: agreement, a confusion matrix, a per-class report, and an honest paragraph on where the classifier fails.

Work straight down the notebook. Each section explains what we are doing and why, then does it. By the last cell the artefact exists and runs.

> ### 🤖 Working with Gemini in this notebook
>
> Colab has Gemini built in. You are **encouraged** to use it — reading and
> directing AI-written code is part of the craft this unit teaches.
>
> **One hard rule:** never paste rows of CRSP data into Gemini (or any other
> AI tool). The licence forbids it. Describe the *schema* instead — column
> names, dtypes, what a row means — which is all an assistant needs to write
> correct code.

## Setup

Run these two cells once, top to bottom, before anything else.

In [ ]:
# Nothing to install: this notebook uses only what Colab already ships
# (pandas, numpy, matplotlib, statsmodels, scipy, sklearn, pyarrow).

In [ ]:
# --- Data -------------------------------------------------------------------
# The unit's datasets are distributed through Blackboard. Download the
# `ACFI20009_data` folder once, put it in your own Google Drive, and this cell
# will find it. Nothing licensed is ever stored in the public repository, and
# the data never leaves your personal Drive.
from pathlib import Path

try:                                    # running on Colab
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/ACFI20009/data")
except ModuleNotFoundError:             # running locally (convenor)
    DATA = Path("2627/data")

assert DATA.exists(), (
    f"Data folder not found at {DATA}.\n"
    "Download it from Blackboard and place it at MyDrive/ACFI20009/data"
)
print("Data:", DATA, "|", len(list(DATA.glob('*.parquet'))), "parquet files")

---

The first stretch of the notebook rebuilds the evaluation pipeline end-to-end: the gold standard, last week's dictionary classifier, and the three ways of scoring one against the other. Everything after that runs your own labels through the same machinery.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
pd.options.display.float_format = "{:.4f}".format
pd.options.display.max_colwidth = None   # show whole sentences, not just the first 50 characters

In [ ]:
gold = pd.read_parquet(DATA / "labeled_sentences.parquet")
print(gold.shape)
print(gold["label_sentiment"].value_counts())

Risk-factor text skews negative/neutral. That imbalance is the reason accuracy is not trusted anywhere in this notebook — worth saying once here and remembering at every metric below.

The baseline is Week 9's dictionary classifier, rebuilt in ten lines:

In [ ]:
import re

lm = pd.read_parquet(DATA / "lm_dictionary.parquet")
pos_words = set(lm.loc[lm["sentiment"] == "positive", "word"].str.lower())
neg_words = set(lm.loc[lm["sentiment"] == "negative", "word"].str.lower())

def lm_label(text: str) -> str:
    words = re.findall(r"[a-z']+", text.lower())
    p = sum(w in pos_words for w in words)
    n = sum(w in neg_words for w in words)
    if p > n:
        return "positive"
    if n > p:
        return "negative"
    return "neutral"

gold["pred_lm"] = gold["text"].map(lm_label)

The confusion matrix comes next. Rows are the truth, columns are what the classifier said, and every off-diagonal cell is a mistake with a name:

In [ ]:
from sklearn.metrics import confusion_matrix

LABELS = ["negative", "neutral", "positive"]
cm = confusion_matrix(gold["label_sentiment"], gold["pred_lm"], labels=LABELS)
pd.DataFrame(cm,
             index=[f"true {l}" for l in LABELS],
             columns=[f"pred {l}" for l in LABELS])

Find the largest off-diagonal cell and say what it actually is: this is not an error *rate* — it is that many specific sentences we can go and read. In ten minutes, you will.

Now the full per-class report, plus kappa against the gold labels:

In [ ]:
from sklearn.metrics import classification_report, cohen_kappa_score

print(classification_report(gold["label_sentiment"], gold["pred_lm"],
                            labels=LABELS, digits=2, zero_division=0))
print(f"kappa vs gold: {cohen_kappa_score(gold['label_sentiment'], gold['pred_lm']):.2f}")

Two questions worth answering from that report before scrolling on: *which single number matters for screening filings for red flags?* — recall on negative; *and for trading on positive tone?* — precision on positive. The metric follows from the decision, never the other way round.

The last step of the pipeline is triage: actually reading the disagreements.

In [ ]:
disagree = gold.loc[gold["label_sentiment"] != gold["pred_lm"],
                    ["sent_id", "text", "label_sentiment", "pred_lm"]]
print(f"{len(disagree)} disagreements")
disagree.head(2)

For each: is this a **classifier error**, a **gold-label error**, or a **guide ambiguity**? Those are the only three options, and each has a different fix (fix the model / fix the label / fix the guide). Section 4 puts that three-way question to work on the LLM's disagreements.

---

### 1. Hand-label 30 sentences (independently)

We draw the 30-sentence sample and print it *without* the gold labels:

In [ ]:
sample30 = (gold.sample(n=30, random_state=10)
                .sort_values("sent_id")
                .reset_index(drop=True))
sheet = sample30[["sent_id", "text"]]
sheet

Working **alone** — no conferring with your partner, no AI assistant — assign each sentence one label from `positive` / `negative` / `neutral`, judged **from the perspective of the filing firm**. Use the Week 10 labelling guide rules: a risk warning about the firm's own business is negative; statements about competitors are judged from the filer's perspective; pure boilerplate with no directional content is neutral. Record your 30 labels in order as a Python list. Do not look at `label_sentiment` until section 3.

There is no answer key to *your* judgement — that is the point — but the mechanics look like this:

In [ ]:
# Your labels, in sample30 row order (0–29). Independence is everything:
# these must be YOUR reads of the sentences, recorded before any comparison.
my_labels = ["negative", "neutral", "neutral", "negative", "positive",
             # ... 25 more ...
             ]
print(f"{len(my_labels)} of 30 labels recorded")   # fill the list in, then re-run
if len(my_labels) >= 30:                           # the checks switch on once the list is full
    assert len(my_labels) == 30
    assert set(my_labels) <= {"positive", "negative", "neutral"}

Working notes a good labeller keeps: (i) decide edge cases by the written guide, not vibes — if the guide is silent, write down the rule you used, because that note becomes an adjudication entry later; (ii) label from the *filer's* perspective throughout ("our competitors reported losses" is not negative for the filer); (iii) resist re-reading earlier answers to make your distribution "look right" — real risk-factor text genuinely skews negative/neutral, and a forced balance is itself a labelling error. Expect to find 4–6 sentences genuinely ambiguous; that is the 75–85% human-agreement ceiling from the lecture showing up in your own sample.

---

### 2. Kappa against your partner

Swap nothing yet: first each of you reads out (or pastes) your label list, then compute raw agreement and Cohen's kappa between the two lists. Interpret the result against the Landis–Koch bands from the lecture. Then — and only then — discuss your disagreements and agree an adjudicated label for each, updating your shared labelling rules as you go.

With your two lists (illustrated here with a synthetic pair so the cell runs offline):

In [ ]:
from sklearn.metrics import cohen_kappa_score

# Illustrative pair — replace with your own and your partner's 30 labels.
mine    = (["negative"] * 12 + ["neutral"] * 14 + ["positive"] * 4)
partner = (["negative"] * 9  + ["neutral"] * 3 + ["neutral"] * 12
           + ["negative"] * 1 + ["positive"] * 1 + ["positive"] * 3
           + ["neutral"] * 1)

raw = np.mean([a == b for a, b in zip(mine, partner)])
print(f"raw agreement: {raw:.2f}   kappa: {cohen_kappa_score(mine, partner):.2f}")

With real labels this lands at raw agreement of roughly 0.7–0.85 and kappa of roughly 0.5–0.75 — the *substantial* band that financial-sentiment studies typically report. Reading the result:

- Kappa well below raw agreement is normal: with a neutral-heavy sample, a big slice of raw agreement is available by chance, and kappa strips it out.
- A pair kappa near the top of the band means your shared labelling rules are tight; near the bottom, list the sentences that split you — they are almost always the same two or three ambiguity types (hedged negatives, competitor statements, conditional language).
- The adjudication step happens **after** kappa, never before. Agreement measured after discussion is not inter-annotator agreement; it is one committee opinion counted twice.
- Common error to avoid: computing kappa when you and your partner labelled different sentence draws. `cohen_kappa_score` will happily return a number; it will mean nothing. Both lists must describe the same 30 sentences in the same order.

---

### 3. Evaluate the LLM against labels it has never seen

The unit ships cached LLM sentiment labels for the gold sample. The API call that produced them (Week 9's schema-first pattern) is shown for reference but not run:

```python
# Needs ANTHROPIC_API_KEY — this is how the cache was produced; do not run in the session.
import json, anthropic
client = anthropic.Anthropic()

def llm_label(sentence: str) -> str:
    prompt = (
        "Classify the sentiment of this 10-K risk-factor sentence from the "
        "perspective of the filing firm. Respond with ONLY a JSON object "
        'like {"label": "negative"}, where label is exactly one of '
        '"positive", "negative", "neutral".\n\nSentence: ' + sentence
    )
    response = client.messages.create(model="claude-opus-4-8", max_tokens=64,
                                      messages=[{"role": "user", "content": prompt}])
    return json.loads(response.content[0].text)["label"]
```

Load the cached run (a seeded stand-in with the real run's error profile, so this file works offline):

In [ ]:
# Stand-in for the cached LLM run: reproduces its measured error profile
# (~3% out-of-schema; errors concentrated in hedged/ambiguous sentences).
# The convenor's real cached file replaces this cell when distributed.
rng = np.random.default_rng(1010)

def cached_llm(true_label: str) -> str:
    u = rng.random()
    if u < 0.03:
        return "mixed"                      # out-of-schema output
    if u < 0.18:                            # classification error
        if true_label == "neutral":
            return "negative"               # the LLM's typical over-call
        return "neutral"
    return true_label

gold["pred_llm"] = [cached_llm(l) for l in gold["label_sentiment"]]
gold["pred_llm"].value_counts()

Three things to establish, in that order: how many of those labels fall outside the legal schema and what to do with them; the confusion matrix, classification report and kappa of `pred_llm` against the gold labels; and whether the LLM beats the dictionary baseline from the opening walkthrough — and on which class. In the session, use *your own* 30 adjudicated labels from section 2 as the truth for your sample rows.

Count the out-of-schema labels first — they are the cheapest possible audit finding:

In [ ]:
bad = ~gold["pred_llm"].isin(LABELS)
print(f"out-of-schema labels: {bad.sum()} of {len(gold)} ({bad.mean():.1%})")

They must be **counted as errors** (or re-queried and the re-query rate reported) — never silently dropped. A label the schema forbids is the classifier failing; deleting the evidence flatters every metric you compute afterwards. The cleanest offline treatment: keep the rows and let them fall off the diagonal.

Now score the full pipeline, out-of-schema warts included:

In [ ]:
cm_llm = confusion_matrix(gold["label_sentiment"], gold["pred_llm"],
                          labels=LABELS + ["mixed"])
print(pd.DataFrame(cm_llm,
                   index=[f"true {l}" for l in LABELS + ["mixed"]],
                   columns=[f"pred {l}" for l in LABELS + ["mixed"]]))

print(classification_report(gold["label_sentiment"], gold["pred_llm"],
                            labels=LABELS, digits=2, zero_division=0))
print(f"kappa vs gold: {cohen_kappa_score(gold['label_sentiment'], gold['pred_llm']):.2f}")

Note the trick: adding `"mixed"` to the label list makes the out-of-schema outputs visible as their own confusion-matrix column (an all-zero *row*, since nothing is truly "mixed") instead of vanishing. In the session, the more valuable version of this comparison uses **your own 30 adjudicated labels** as the truth for your sample rows — labels the LLM cannot have been tuned on, which answers the contamination worry directly.

That leaves the comparison itself: kappas side by side, and the per-class rows of the two reports — the dictionary from the opening walkthrough, the LLM from the report just printed. The LLM comes out ahead on kappa: its error profile is thinner across the board, where the dictionary's errors are spread over several cells of its confusion matrix (true-neutral boilerplate pushed to negative *and* to positive, true negatives read as neutral). But the LLM's score must carry the out-of-schema column: if counting `"mixed"` as errors closes the gap, say so. The exact figures depend on the cached run; the required *reading* does not: name the class where the gap lives, and state whether the winner's advantage survives honest accounting.

---

### 4. Triage two disagreements by hand

Two sentences where `pred_llm` disagrees with the gold label, pulled out with `gold.loc[gold["label_sentiment"] != gold["pred_llm"]]`. For each one we read the sentence and classify the disagreement as (i) a **model error**, (ii) a **gold-label error**, or (iii) a **guide ambiguity** — and state the fix each diagnosis implies.

In [ ]:
triage = gold.loc[gold["label_sentiment"] != gold["pred_llm"],
                  ["sent_id", "text", "label_sentiment", "pred_llm"]]
triage.head(2)

There is no mechanical answer here; what triage produces is a *reasoning pattern*, and it goes like this for each sentence:

1. **Re-read the sentence against the labelling guide**, as if labelling it fresh. Write down your own label before looking at either existing one.
2. **If your fresh label matches the gold label**, the classifier is wrong → **model error**. Fix: note the failure pattern (negation? perspective? hedging?) — if it recurs, the prompt needs an explicit rule or an example, and the change must be re-validated on the frozen set.
3. **If your fresh label matches the classifier**, the gold label is wrong → **gold-label error**. Fix: correct the gold set through the adjudication process (not by quietly editing a file), and expect your measured accuracy to *rise* — some of the classifier's "errors" were the answer key's.
4. **If you genuinely cannot decide**, it is a **guide ambiguity**. Fix: neither the model nor the label — the guide gains a new edge-case rule, and the sentence is re-labelled under it.

Each diagnosis has a *different* fix, which is why triage cannot be skipped: retraining a model to match wrong gold labels, or "correcting" gold labels to match a wrong model, both make the pipeline worse while improving the score. One cleanly documented triage case — the sentence, the diagnosis, the fix — is precisely the shape of evidence the audit workflow from the lecture asks you to bank.

---

### 5. Verify the AI

A student asked an AI assistant to "evaluate my LLM classifier against the gold labels and tell me if it's safe to use for screening filings for red flags". The assistant produced the following code and conclusion:

In [ ]:
# --- AI assistant's evaluation (verbatim) ---
ok = gold["pred_llm"].isin(LABELS)          # remove malformed labels: they can't be scored
print(classification_report(gold.loc[ok, "pred_llm"],
                            gold.loc[ok, "label_sentiment"],
                            labels=LABELS, digits=2, zero_division=0))

> *"Recall on the negative class is strong, so the classifier is safe to deploy as a red-flag screen. Malformed labels were removed as they cannot be scored."*

The code runs without error and the numbers are real, and the evaluation is still wrong: there is one central flaw and at least one further problem. Worth pausing to find them before reading on — the diagnosis, the corrected version, and the fate of the assistant's conclusion all follow.

**The central flaw: the arguments are swapped.** The signature is `classification_report(y_true, y_pred)`, and the assistant passed the *predictions* first. Swapping the arguments transposes the confusion matrix, and therefore **exchanges precision and recall for every class**. The "strong negative recall" the assistant praised is actually the classifier's negative *precision* — a different quantity answering a different question. For a red-flag screen, the number that matters is true recall on negative: *of the genuinely negative sentences, how many did we find?* The swap reports instead: *of the sentences we flagged, how many were right?* — exactly the mix-up from the lecture's AI checkpoint, wearing new clothes.

**The further problem: silently deleting the out-of-schema labels.** Every `"mixed"` output is a classifier failure on a real sentence; filtering those rows shrinks the denominator and flatters every metric. They must be scored as errors (or re-queried, with the rate reported).

The corrected evaluation:

In [ ]:
print(classification_report(gold["label_sentiment"],   # y_true first
                            gold["pred_llm"],          # y_pred second
                            labels=LABELS, digits=2, zero_division=0))

Each class's precision and recall swap places relative to the assistant's output — compare the two negative rows side by side — and the metrics dip slightly further once the out-of-schema rows count against the classifier. Whether "safe to deploy" survives now rests on the *true* negative recall — and note the final trap in the assistant's sentence: "safe to deploy" is a decision that belongs to the analyst who owns the error costs, not to the model being evaluated, and not to the assistant evaluating it. The lecture's rule stands: never accept a self-reported score you have not recomputed with the arguments the right way round.

---

### 6. Exhibit workshop: Figure 1 from your own pipeline

The last piece of the Validation File is the exhibit itself. Using this week's pipeline we build a first candidate for a client-facing figure — firm-year negative-tone shares from the EDGAR sample, as a chart a reader could act on *without you in the room* — and then put it through the exhibit test: (i) does the title state the claim rather than the topic? (ii) are the axes labelled with units? (iii) is the data source and method stated? (iv) could the client act on this figure alone?

First the firm-year tone measure, then the exhibit:

In [ ]:
sents = pd.read_parquet(DATA / "edgar_10k_sentences.parquet")
sents["pred_lm"] = sents["text"].map(lm_label)
sents["is_neg"] = sents["pred_lm"] == "negative"

tone = (sents.groupby(["company", "filing_year"])
             .agg(n_sentences=("sent_id", "size"),
                  neg_share=("is_neg", "mean")))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4.5))
(tone["neg_share"].unstack("filing_year")
     .plot.barh(ax=ax, color=["#0F6BBA", "#0F766E", "#9CA3AF", "#D97706"]))
ax.set_title("Risk-factor tone varies more across firms than across years")
ax.set_xlabel("Share of Item 1A sentences classified negative (LM dictionary)")
ax.set_ylabel("")
ax.legend(title="Filing year")
fig.text(0, -0.04,
         "Source: SEC EDGAR 10-K Item 1A sentences; Loughran-McDonald "
         "classification validated against hand-checked labels.", fontsize=8)
plt.show()

The figure against the exhibit test:

- **Title states the claim** — "Risk-factor tone varies more across firms than across years", not "Negative tone by firm". If your computed chart contradicts that claim, change the title to what the chart *does* show; never keep a title the data does not support.
- **Axis with units** — "share of Item 1A sentences classified negative" says what the number is *and* how it was made; a bare "neg_share" fails a client immediately.
- **Source and method in the caption** — including the validation claim, which is what separates an exhibit from decoration: a client can ask "how do you know the classifier is right?" and the caption already answers where to look.
- **Could the client act on it alone?** Almost — the honest reservation, which belongs in the accompanying text, is sample size: with ~10 firms and 2–3 filing years this figure *describes* tone, it does not establish a return-predicting signal. Saying so is not weakness; it is the difference between analysis and advertising.

The most common fixable faults are topic-titles, unlabelled y-axes, and legends left with the raw `filing_year` column name as their title — all thirty-second fixes with outsized effect on how professional the exhibit reads.

## Before you leave

The Validation File now exists: agreement with a partner, a confusion matrix, a per-class report, two triaged disagreements with their diagnoses, and an exhibit whose caption states the method and the validation behind it. That is the document you would attach to a model-risk review — and the honest paragraph on where the classifier fails is the part a reviewer reads first.